# Tier 4.2 — Agentic ReAct Experiments (Azure GPU)

**BSARD RAG Thesis | RQ1 | Backbone: react_bm25 (bm25_tuned_k11.5_b0.25)**

## Before running — prerequisites

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC6ads_A10_v4` (preferred) or `NC4as_T4_v3`
2. **Set Cell 0** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. **ReAct implementation must be complete** in the repo:
   - `retrieval/agentic/react.py` and `retrieval/agentic/tools.py`
   - `scripts/evaluation/tier4/run_react_experiments.py`
4. Run cells top to bottom

### How to generate the Container SAS URL
Azure Portal → Storage Accounts → *your account* → Containers → `bsard-data`
→ `...` → **Generate SAS** → Permissions: **Read + List** → Expiry: 1 year
→ Generate → copy the **Blob SAS URL** (full `https://...` URL, not just the token)

## Expected execution times (T4 GPU, includes D1 LLaMA post-hoc re-ranking)

| Phase | Expected time |
|---|---|
| Setup (Cells 0–7) | ~20 min |
| TEST experiment (Cell 9) | ~2 h |
| **Total** | **~2.5 h** |

## Hyperparameters (hardcoded — no tuning)

| Parameter | Value | Rationale |
|---|---|---|
| `max_steps` | 8 | More reasoning rounds, larger observation pool |
| `top_k_shown` | 10 | Wider search per step, better coverage |
| `overlap_threshold` | 0.7 | Empirically sound |
| `max_article_tokens` | 1000 | Matches T4.0 D1 token budget, better re-rank quality |

## Resuming after interruption
The test result file is written immediately and skipped on re-run.
Commit with Cell 11 after the run.

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
# Set GITHUB_TOKEN and AZURE_CONTAINER_SAS_URL before running any other cell.

GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3 or NC6ads_A10_v4')
    print(result.stderr)


In [ ]:
# ── Cell 2: Install Ollama and pull llama3.1:8b (~10 min on first run) ────────
import json, os, subprocess, time, urllib.request

OLLAMA_LOG = '/tmp/ollama_server.log'

def model_available() -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            return any('llama3.1' in m['name'] for m in json.loads(r.read()).get('models', []))
    except Exception:
        return False

# ── Install Ollama if not present ─────────────────────────────────────────────
if not os.path.exists('/usr/local/bin/ollama'):
    print('Installing Ollama via official script...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True
    )
    print('Ollama installed.')
else:
    print('Ollama already installed.')

# ── Start server (if not already running) ─────────────────────────────────────
if not model_available():
    print('Starting Ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        env={
            **os.environ,
            'HOME': '/root',
            'OLLAMA_NUM_GPU': '99',         # use all GPU layers
            'OLLAMA_FLASH_ATTENTION': '1',  # faster on A10/T4
            'OLLAMA_HOST': '0.0.0.0:11434',
        },
        stdout=open(OLLAMA_LOG, 'w'),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)

    if not model_available():
        print('Pulling llama3.1:8b (~4.7 GB, ~5-10 min)...')
        subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
        time.sleep(3)

# ── Verify ─────────────────────────────────────────────────────────────────────
resp   = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
models = [m['name'] for m in json.loads(resp.read()).get('models', [])]
print('Available models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'
print('Ollama ready.')


In [ ]:
# ── Cell 3: Download data from Azure Blob Storage ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'], check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path

OUTPUT_DIR = Path(REPO_DIR) / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# blob name in container → local destination directory
downloads = {
    'bsard_articles_dedup.parquet':                               OUTPUT_DIR,
    'bsard_corpus.db':                                            OUTPUT_DIR,
}

client = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)

for blob_name, dest_dir in downloads.items():
    dest_path = Path(dest_dir) / blob_name
    if dest_path.exists():
        print(f'  Already exists: {blob_name} ({dest_path.stat().st_size / 1e6:.1f} MB)')
        continue
    print(f'  Downloading {blob_name} ...', end='', flush=True)
    with open(dest_path, 'wb') as f:
        client.get_blob_client(blob_name).download_blob().readinto(f)
    print(f' done ({dest_path.stat().st_size / 1e6:.1f} MB)')

print('\nAll data files ready.')


In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    # Re-embed token so pull works across VM sessions (T4.0 / T4.1 pattern)
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')


In [ ]:
# ── Cell 5: Install Python dependencies (~5 min) ──────────────────────────────
import subprocess, sys

cmds = [
    ([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
     'requirements.txt'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'],
     'azure-storage-blob'),
    ([sys.executable, '-m', 'pip', 'install', '-q',
      'langgraph>=0.1.0', 'langchain-core>=0.2.0', 'requests'],
     'langgraph + langchain-core + requests'),
]
for cmd, label in cmds:
    print(f'  {label} ...', end='', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})')
    if r.returncode != 0:
        print(r.stderr[-200:])

# bsard_evaluation — editable local package from the RQ3 repo (required by evaluation/runner.py)
import os
# bsard_evaluation lives in the same mono-repo (RQ3_Autonomous_Evaluation),
# already cloned above — just install it editable.
RQ3_DIR = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'
print('  bsard_evaluation ...', end='', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', RQ3_DIR],
                   capture_output=True, text=True)
print(' OK' if r.returncode == 0 else (f' WARN({r.returncode})\n' + r.stderr[-200:]))

# spaCy French model
import spacy as _spacy
_sv = _spacy.__version__
_base = 'https://github.com/explosion/spacy-models/releases/download'
_whl  = f'fr_core_news_lg-{_sv}/fr_core_news_lg-{_sv}-py3-none-any.whl'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl}'],
    capture_output=True, text=True
)
if r.returncode != 0:
    _sv2 = '.'.join(_sv.split('.')[:2]) + '.0'
    _whl2 = f'fr_core_news_lg-{_sv2}/fr_core_news_lg-{_sv2}-py3-none-any.whl'
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl2}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError('spaCy fr_core_news_lg install failed:\n' + r.stderr[-400:])

import spacy
spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK — fr_core_news_lg loaded')

In [ ]:
# -- Cell 6: Verify spaCy French model ---------------------------------------
# fr_core_news_lg was installed via the direct pip wheel URL in Cell 5.
# Using 'spacy download' here can 404 on GitHub releases (T4.0/T4.1 lesson).
import spacy
spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK -- fr_core_news_lg loaded')

In [ ]:
# ── Cell 7: Pre-flight checks + LLM latency benchmark ────────────────────────
import os, sys, time, json
import requests as _req
from pathlib import Path

os.chdir(REPO_DIR)

# ── File checks ───────────────────────────────────────────────────────────────
for p in [
    Path('output/bsard_articles_dedup.parquet'),
    Path('output/bsard_corpus.db'),
    Path('evaluation/data/fewshot_examples.json'),
]:
    print(f'  {"OK     " if p.exists() else "MISSING"}  {p}')

# ── Ollama alive ──────────────────────────────────────────────────────────────
try:
    r      = _req.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'\nOllama alive. Models: {models}')
except Exception as e:
    raise RuntimeError(f'Ollama not reachable: {e}')

# ── LLM latency benchmark (3 calls) ──────────────────────────────────────────
def llm_generate(prompt, max_tokens=5):
    t0   = time.perf_counter()
    resp = _req.post('http://localhost:11434/api/generate', json={
        'model': 'llama3.1:8b', 'prompt': prompt, 'stream': False,
        'options': {'temperature': 0.0, 'num_predict': max_tokens},
    }, timeout=1800)
    return resp.json()['response'].strip(), (time.perf_counter() - t0) * 1000

TEST_PROMPT = (
    'Question : Quelles sont les conditions pour obtenir un congé parental ?\n\n'
    'Passage : Le congé parental est accordé aux travailleurs salariés ayant un '
    'enfant de moins de 12 ans.\n\n'
    'Le passage est-il pertinent pour répondre à la question ?\n'
    'Répondez uniquement par « Oui » ou « Non ».\n\nPertinent :'
)
lats = []
for i in range(3):
    resp, lat = llm_generate(TEST_PROMPT)
    lats.append(lat)
    print(f'  Call {i+1}: {resp!r:10s}  {lat:.0f} ms')

mean_lat = sum(lats) / len(lats)
print(f'\nMean latency: {mean_lat:.0f} ms  →  ', end='')
if mean_lat < 5000:
    print('GPU confirmed (fast).')
elif mean_lat < 30000:
    print('MARGINAL — verify GPU in Cell 1.')
else:
    print('WARNING: likely on CPU. Re-check nvidia-smi.')

# D1 at 1000 tokens dominates runtime. Calibrated from T4.0 run on same hardware:
#   1000-tok binary call: 0.90s warm on Tesla T4
#   generate_step: ~0.6s (shorter scratchpad prompt)
D1_CALL_S = 0.90   # T4.0 observed: 1000-tok LLaMA binary call on T4
GEN_CALL_S = 0.60  # generate_step estimate (~1500-tok scratchpad + system)
D1_POOL_EST = 25   # estimated mean pool after max_steps=8, top_k_shown=10
test_q = 222
est_s = D1_POOL_EST * D1_CALL_S + 8 * GEN_CALL_S
print(f'\n--- Estimated experiment time (T4.2 ReAct) ---')
print(f'  ~{D1_POOL_EST} D1 calls x {D1_CALL_S:.2f}s  +  8 generate calls x {GEN_CALL_S:.2f}s  =  ~{est_s:.0f}s/query')
print(f'  TEST run: ~{test_q * est_s / 60:.0f} min  ({test_q} questions)')
if mean_lat > 1000:
    revised = test_q * (D1_POOL_EST * mean_lat / 1000 + 8 * 0.6) / 60
    print(f'  WARNING: LLM is {mean_lat:.0f}ms warm (expected ~400ms). '
          f'GPU may not be active. Revised estimate: ~{revised:.0f} min.')
    print('  Re-check Cell 1 (nvidia-smi) before starting Cell 9.')


In [ ]:
# ── Cell 8: Hardcoded hyperparameters ──────────────────────────────────
# No tuning — parameters fixed based on latency budget analysis.
# Budget: ~30 s/query × 222 test questions ≈ 2 h on Tesla T4.
MAX_STEPS          = 5
TOP_K_SHOWN        = 10
OVERLAP_THRESHOLD  = 1.1   # disables D2 guard (Jaccard can never reach 1.1)
MAX_ARTICLE_TOKENS = 200

print('Hardcoded hyperparameters:')
print(f'  max_steps          = {MAX_STEPS}')
print(f'  top_k_shown        = {TOP_K_SHOWN}')
print(f'  overlap_threshold  = {OVERLAP_THRESHOLD}')
print(f'  max_article_tokens = {MAX_ARTICLE_TOKENS}')
print(f'\nEstimated test runtime: ~{222 * 30 / 60:.0f} min ({222} questions x ~30 s/query)')

In [ ]:
# # Delete old files so Cell 9 won't be skipped

# import os
# f = f'{REPO_DIR}/output/results/agentic/ReAct/react_bm25_test.json'
# if os.path.exists(f):
#     os.remove(f)
#     print('Deleted — ready to re-run.')
# else:
#     print('File not found.')


In [ ]:
# ── Create react_hyperparams.json (skips tuning, uses hardcoded values) ──
import json
from pathlib import Path

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/ReAct')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

hparams = {
    "max_steps":         MAX_STEPS,
    "top_k_shown":       TOP_K_SHOWN,
    "overlap_threshold": OVERLAP_THRESHOLD,
    "recall10_val":      None,
    "fraction_queries_terminated_by_limit_best": None,
    "grid_results":      [],
    "tuning_variant":    "react_bm25",
    "val_questions":     None,
}
out = RESULTS_DIR / 'react_hyperparams.json'
out.write_text(json.dumps(hparams, indent=2))
print(f'Written: {out}')


In [ ]:
# ── Cell 9: TEST experiment ─────────────────────────────────────
# Writes: react_bm25_test.json. Skipped if already done.
# Expected: ~2 h (max_steps=8, top_k_shown=10, max_article_tokens=1000)
import json, os, subprocess, sys, time

print('Unloading Ollama model to free RAM for BM25 tokenization...')
subprocess.run(['ollama', 'stop', 'llama3.1:8b'], capture_output=True)
print('Ollama model unloaded.')

print('*** TEST SPLIT ***')
print(f'max_steps={MAX_STEPS}  top_k_shown={TOP_K_SHOWN}  overlap_threshold={OVERLAP_THRESHOLD}  max_article_tokens={MAX_ARTICLE_TOKENS}')

for variant in ['bm25']:
    out = f'{REPO_DIR}/output/results/agentic/ReAct/react_{variant}_test.json'
    if os.path.exists(out):
        r10 = json.loads(open(out).read())['metrics'].get('Recall@10', 0)
        print(f'  [{variant:5s} test]  already done  R@10={r10:.4f}')
        continue
    print(f'\n  [{variant:5s} test]  starting ...')
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, 'scripts/evaluation/tier4/run_react_experiments.py',
         '--split', 'test', '--variant', variant,
         '--max-steps', str(MAX_STEPS),
         '--top-k-shown', str(TOP_K_SHOWN),
         '--overlap-threshold', str(OVERLAP_THRESHOLD),
         '--max-article-tokens', str(MAX_ARTICLE_TOKENS)],
        cwd=REPO_DIR, timeout=43200,
    )
    print(f'  [{variant:5s} test]  done in {(time.time()-t0)/60:.1f} min  exit={result.returncode}')
    if result.returncode != 0:
        print(f'  WARNING: non-zero exit for {variant} test')

print('\nTest experiment complete (or skipped).')

In [ ]:
# import os
# from pathlib import Path

# result_file = Path(f'{REPO_DIR}/output/results/agentic/ReAct/react_bm25_test.json')
# traces_file = Path(f'{REPO_DIR}/output/results/agentic/ReAct/react_bm25_test_traces.json')

# print('Result:', result_file.exists())
# print('Traces:', traces_file.exists())


In [ ]:
# ── Cell 10: Final results summary ─────────────────────────────────
import json
from pathlib import Path

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/ReAct')

print(f'{"Experiment":<35} {"R@10":>7} {"R@100":>7} {"MRR@10":>8} {"Wall(min)":>10}')
print('=' * 73)
for variant in ['bm25']:
    path = RESULTS_DIR / f'react_{variant}_test.json'
    if not path.exists():
        print(f'  react_{variant}_test  (not yet run)')
        continue
    r    = json.loads(path.read_text())
    m    = r.get('metrics', {})
    wall = r.get('total_experiment_wall_clock_s', 0) / 60
    print(f'  {r["experiment_id"]:<35}'
          f'{m.get("Recall@10",0):>7.4f}'
          f'{m.get("Recall@100",0):>7.4f}'
          f'{m.get("MRR@10",0):>8.4f}'
          f'{wall:>10.1f}')

print('\n--- Hyperparameters ---')
print(f'  max_steps={MAX_STEPS}  top_k_shown={TOP_K_SHOWN}  overlap_threshold={OVERLAP_THRESHOLD}  max_article_tokens={MAX_ARTICLE_TOKENS}')

print('\n--- Loop stats ---')
for variant in ['bm25']:
    path = RESULTS_DIR / f'react_{variant}_test.json'
    if not path.exists():
        continue
    r    = json.loads(path.read_text())
    loop = r.get('agent_loop_stats', {})
    bd   = r.get('latency_breakdown_ms_mean', {})
    lim  = loop.get('fraction_queries_terminated_by_limit', 0)
    print(f'  {variant}: mean_steps={loop.get("mean_steps_per_query",0):.1f}  '
          f'fin%={loop.get("fraction_queries_converged_finish",0):.1%}  '
          f'term_lim%={lim:.1%}')
    print(f'         llm_gen={bd.get("llm_generate",0):.0f}ms  '
          f'retrieval={bd.get("retrieval",0):.0f}ms  '
          f'rerank={bd.get("rerank_posthoc",0):.0f}ms')
    if lim > 0.30:
        print(f'    *** WARNING: {lim:.0%} queries hit step limit')

In [ ]:
# -- Cell 11: Per-step recall analysis -------------------------------------------
# For each step t=1..MAX_STEPS, reconstructs the cumulative observed-ID pool and
# computes:
#   Recall@10          : top-10 by observation order vs ground truth
#   Recall@all_observed: fraction of relevant articles anywhere in the pool
#                        (ceiling recall -- what perfect re-ranking could achieve)
#   MRR@10             : MRR using observation-order ranking
#
# Reads : react_bm25_test_traces.json  (saved by run_react_experiments.py)
#         react_bm25_test.json         (for final D1 metrics)
# Writes: react_bm25_test_step_recall.json
import json, sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from evaluation.split import load_questions
from evaluation.metrics import evaluate

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/ReAct')
TRACES_PATH = RESULTS_DIR / 'react_bm25_test_traces.json'
OUTPUT_PATH = RESULTS_DIR / 'react_bm25_test_step_recall.json'
RESULT_PATH = RESULTS_DIR / 'react_bm25_test.json'

if not TRACES_PATH.exists():
    print('ERROR: traces file not found. Re-run Cell 9 first.')
else:
    traces_data  = json.loads(TRACES_PATH.read_text())
    questions    = load_questions(subset='test')
    ground_truth = {q['question_id']: q['relevant_article_ids'] for q in questions}
    n_queries    = len(traces_data)

    per_step = {}
    for step_t in range(1, MAX_STEPS + 1):
        step_results = {}
        pool_sizes   = []
        for entry in traces_data:
            qid  = entry['question_id']
            pool, seen = [], set()
            for t_entry in entry['trace'][:step_t]:
                for aid in t_entry.get('ids_returned', []):
                    if aid not in seen:
                        pool.append(aid)
                        seen.add(aid)
            step_results[qid] = pool
            pool_sizes.append(len(pool))

        m = evaluate(step_results, ground_truth)

        # Recall@all_observed: mean fraction of relevant articles in the pool
        total_recall_all = 0.0
        for entry in traces_data:
            qid     = entry['question_id']
            pool_s  = set(step_results[qid])
            rel     = set(ground_truth.get(qid, []))
            if rel:
                total_recall_all += len(pool_s & rel) / len(rel)
        mean_recall_all = total_recall_all / n_queries

        per_step[step_t] = {
            'Recall@10':           round(m.get('Recall@10', 0.0), 6),
            'Recall@all_observed': round(mean_recall_all,          6),
            'MRR@10':              round(m.get('MRR@10',    0.0), 6),
            'mean_pool_size':      round(sum(pool_sizes) / n_queries, 2),
        }

    # Final D1 metrics
    final_d1 = {}
    if RESULT_PATH.exists():
        fm = json.loads(RESULT_PATH.read_text()).get('metrics', {})
        final_d1 = {
            'Recall@10': round(fm.get('Recall@10', 0.0), 6),
            'MRR@10':    round(fm.get('MRR@10',    0.0), 6),
        }

    # Print table
    print(f'{"Step":>6}  {"R@10":>8}  {"R@all":>8}  {"MRR@10":>8}  {"Pool":>6}')
    print('-' * 50)
    for step_t in range(1, MAX_STEPS + 1):
        m = per_step[step_t]
        print(f'{step_t:>6}  {m["Recall@10"]:>8.4f}  '
              f'{m["Recall@all_observed"]:>8.4f}  '
              f'{m["MRR@10"]:>8.4f}  '
              f'{m["mean_pool_size"]:>6.1f}')
    if final_d1:
        print('-' * 50)
        print(f'{"D1":>6}  {final_d1["Recall@10"]:>8.4f}  '
              f'{"":>8}  {final_d1["MRR@10"]:>8.4f}')

    output = {
        'experiment_id': 'react_bm25_test',
        'n_queries':      n_queries,
        'max_steps':      MAX_STEPS,
        'per_step':       per_step,
        'final_d1':       final_d1,
    }
    OUTPUT_PATH.write_text(json.dumps(output, indent=2, ensure_ascii=False))
    print(f'\nSaved: {OUTPUT_PATH}')

In [ ]:
# -- Cell 12: Commit and push results to GitHub -----------------------------------
# Safe to run after Cells 9 and 11 complete.
import os, subprocess
from pathlib import Path

GIT_NAME  = 'MariusPasch'
GIT_EMAIL = 'paschalidismarios@gmail.com'

def git(args):
    return subprocess.run(['git'] + args, cwd=REPO_DIR, capture_output=True, text=True)

git(['config', 'user.email', GIT_EMAIL])
git(['config', 'user.name',  GIT_NAME])
git(['remote', 'set-url', 'origin', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'])

files_to_stage = [
    'output/results/agentic/ReAct/react_bm25_test.json',
    'output/results/agentic/ReAct/react_bm25_test_traces.json',
    'output/results/agentic/ReAct/react_bm25_test_step_recall.json',
]
staged = [f for f in files_to_stage if Path(f'{REPO_DIR}/{f}').exists()]
for f in staged:
    git(['add', f])

print(f'Staged {len(staged)} file(s):')
for f in staged:
    print(f'  {f}')

if not staged:
    print('No result files found to commit.')
else:
    status = git(['status', '--short']).stdout.strip()
    if not status:
        print('Nothing new to commit.')
    else:
        commit = git(['commit', '-m',
            'T4.2 ReAct results -- LLaMA 3.1 8B, D1 post-hoc rerank, per-step recall, Azure GPU\n\n'
            ''])
        print('Commit:', commit.stdout.strip() or commit.stderr.strip())
        push = git(['push', 'origin', 'main'])
        print('Pushed.' if push.returncode == 0 else f'Push failed: {push.stderr[-400:]}')

In [ ]:
# import subprocess

# def git(args):
#     return subprocess.run(['git'] + args, cwd=REPO_DIR, capture_output=True, text=True)

# # Check if output/ is ignored
# print(git(['check-ignore', '-v', 'output/results/agentic/ReAct/react_bm25_test.json']).stdout)

# # Force-add and commit
# for f in [
#     'output/results/agentic/ReAct/react_bm25_test.json',
#     'output/results/agentic/ReAct/react_bm25_test_traces.json',
#     'output/results/agentic/ReAct/react_bm25_test_step_recall.json',
# ]:
#     git(['add', '-f', f])

# status = git(['status', '--short']).stdout.strip()
# print('Status:', status)
# if status:
#     commit = git(['commit', '-m',
#         'T4.2 ReAct results — overlap=1.1, max_steps=5, top_k=10, zero-shot D1\n\n'
#         ''])
#     print(commit.stdout.strip())
#     push = git(['push', 'origin', 'main'])
#     print('Pushed.' if push.returncode == 0 else push.stderr[-300:])
# else:
#     print('Nothing to commit — files may already be committed.')


In [ ]:
r = git(['pull', '--rebase', 'origin', 'main'])
print(r.stdout, r.stderr)
r = git(['push', 'origin', 'main'])
print('Pushed.' if r.returncode == 0 else r.stderr[-300:])
